# Patient Distribution Mapping — Visualization (Member 2 / Type B)

**Business Question:** Patient Distribution Mapping
**Type A partner:** Member 1
**Input:** `data/processed/department_patient_distribution.csv` (prepared by Member 1)
**Output:** `patient_distribution_map.html` — interactive Folium map for the executive dashboard


In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap

dist = pd.read_csv("../data/processed/department_patient_distribution.csv")
dist.head()

In [ ]:
total_patients = dist["patient_count"].sum()
dist["patient_percentage"] = (dist["patient_count"] / total_patients) * 100
dist = dist.sort_values("patient_count", ascending=False).reset_index(drop=True)

top5_ids = set(dist.head(5)["department_id"])
dist.head(5)[["department_id", "department_name", "patient_count", "patient_percentage"]]

## Build the map

- Base layer: OpenStreetMap centered on the mean facility coordinates
- **Heatmap layer**: patient volume weighted density across all 20 departments
- **Marker layer**: one circle marker per department, sized by patient count, with a popup
  showing patient count / admission count / share of total
- Top-5 volume departments are highlighted in red


In [ ]:
center_lat = dist["latitude"].mean()
center_lon = dist["longitude"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles="OpenStreetMap")

heat_data = [[row.latitude, row.longitude, row.patient_count] for row in dist.itertuples()]
HeatMap(heat_data, radius=28, blur=20, max_zoom=13, min_opacity=0.35).add_to(m)

min_c, max_c = dist["patient_count"].min(), dist["patient_count"].max()

for row in dist.itertuples():
    is_top = row.department_id in top5_ids
    color = "#d62728" if is_top else "#1f77b4"
    radius = 6 + (row.patient_count - min_c) / (max_c - min_c) * 16

    popup = folium.Popup(
        f"<b>{row.department_name}</b><br>"
        f"Patients: {row.patient_count}<br>"
        f"Admissions: {row.admission_count}<br>"
        f"Share: {row.patient_percentage:.1f}%",
        max_width=250,
    )

    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=radius,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.75,
        tooltip=f"{row.department_name} — {row.patient_count} patients",
        popup=popup,
    ).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m

## Findings summary (for Documentation Lead / final presentation)

- **Highest patient volume:** Emergency (335 patients, 6.7% of total)
- **Lowest patient volume:** Gynecology (159 patients, 3.2% of total)
- **Top-5 departments** (Emergency, Oncology, Orthopedics, Endocrinology, Pulmonology) together account for
  roughly 32% of all patient volume, clustered in the northeast/central part of the coverage area.
- Volume is fairly evenly spread across the remaining 15 departments (3–5% each), i.e. no single extreme outlier
  beyond the Emergency/Oncology pair.
- **Assumption carried over from Member 1 / GEOCODE_README:** coordinates are department/facility-level
  benchmarks, not patient home-address data — the map shows *where patients were treated*, not where they live.

## Output

Map exported to `patient_distribution_map.html` for embedding in the Plotly/Dash executive dashboard
(hand-off to Member 8 — Dashboard Integration Lead).


In [ ]:
m.save("../outputs/patient_distribution_map.html")
print("Saved: outputs/patient_distribution_map.html")